# NFL Player Contact Detection - MultiShot MultiPose Estimation using YOLOv8 & Mediapipe

# Problem Statement

## Goal of the Competition
The goal of this competition is to detect external contact experienced by players during an NFL football game. You will use video and player tracking data to identify moments with contact to help improve player safety.
​
## Context
The National Football League (NFL) has teamed up with Amazon Web Services (AWS) to strengthen its commitment to predict player injuries. The NFL aspires to have the best injury surveillance and mitigation program in any sport. With your machine learning and computer vision skills, you can help the NFL accurately identify when players experience contact throughout a football play.
​
In prior years, the NFL challenged the Kaggle community to create helmet impact detection and identification algorithms. This year the NFL looks to automatically identify all moments when players experience contact. This competition will be successful if we can reliably detect moments when players are in contact with one another and when a player’s body is in contact with the ground.
​
Currently, the NFL uses its tracking system to monitor a large number of statistics about players’ load during the season. The league has a solution that predicts contact between players, but it only leverages the player tracking data. This competition hopes to improve the predictive power by including video in addition to tracking data. Categorizing ground contact will also provide a more comprehensive view of impacts, improving analysis for player health and safety.
​
More accurate data is an important step toward the NFL’s injury surveillance and mitigation goals. With complete contact detection, the league can identify correlations between certain types of contact and injury, a contributor to future prevention. Your efforts could help mitigate unsafe situations to reduce injury to all players.
​
The National Football League is America's most popular sports league. Founded in 1920, the NFL developed the model for the successful modern sports league and is committed to advancing progress in the diagnosis, prevention, and treatment of sports-related injuries. This competition is part of the Digital Athlete, a joint effort between the NFL and AWS to build a virtual, 360-degree representation of an NFL player’s experience. The Digital Athlete hopes to generate a precise picture of what they need when it comes to preventing and recovering from injuries while performing at their best. Health and safety efforts include support for independent medical research and engineering advancements as well as a commitment to work to better protect players and make the game safer, including enhancements to medical protocols and improvements to how our game is taught and played. For more information about the NFL's health and safety efforts, please visit the [NFL Player Health and Safety website](https://www.nfl.com/playerhealthandsafety/).
​
## Evaluation
Submissions are evaluated on [Matthews Correlation Coefficient](https://en.wikipedia.org/wiki/Phi_coefficient) between the predicted and actual contact events.
​
​
$$MCC = \frac{TP * TN - FP * FN}{\sqrt{(TP + FP)(TP + FN)(TN + FP)(TN + FN)}}$$
​
For every allowable `1contact_id` (formed by concatenating the `game_play_step_player1_player2`), you must predict whether the involved players are in contact at that moment in time. sample_submission.csv provides the exhaustive list of `contact_ids`. Note that the ground, denoted as player G, is included as a possible contact in place of player2. The player with the lower id is always listed first in the `contact_id`.
​
The file should contain a header and have the following format:
​
```
contact_id,contact
58168_003392_0_38590_43854,0
58168_003392_0_38590_41257,1
58168_003392_0_38590_41944,0
etc.
```

# Import dependencies

In [1]:
# install mediapipe
!pip install mediapipe
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.0/33.0 MB 12.5 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 1.12
    Uninstalling flatbuffers-1.12:
      Successfully uninstalled flatbuffers-1.12
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-io 0.21.0 requires tensorflow-io-gcs-filesystem==0.21.0, which is not installed.
tensorflow 2.6.4 requires flatbuffers~=1.12.0, but you have flatbuffers 23.1.21 which is incompatible.
tensorflow 2.6.4 requires h5py~=3.1.0, but you have h5py 3.7.0 which is incompatible.
tensorflow 2.6.4 requires numpy~=1.19.2, but you have numpy 1.21.6 which is incompatible.
tensorflow 2.6.4 requires typing-extensions<3.11,>=3.7, but you have typing-extensions 4.1.1 which is incompatible.
tensorflow-transform 1.9.0 requires tensorflow!=2.0.*,!=2.1.*,!=2.2.*,!=2.3.*,!=2.4.*,!=

In [2]:
import ultralytics
ultralytics.checks()

Ultralytics YOLOv8.0.22 🚀 Python-3.7.12 torch-1.11.0 CUDA:0 (Tesla P100-PCIE-16GB, 16281MiB)
Setup complete ✅ (2 CPUs, 15.6 GB RAM, 4177.2/8062.4 GB disk)


In [3]:
# import dependencies
import os
import subprocess
import IPython
from IPython.display import Video, display

import numpy as np
import pandas as pd

import cv2
import mediapipe as  mp

# YOLOv8 + Mediapipe MultiShot MultiPose Estimation

## Approach
- We use YOLOv8 object detection to first find the bounding boxes of all the persons in the video.
- We use the bounding boxes on to indiviudally crop each frame to include only one person.
- We run MediaPipe single pose estimation on the cropped frame to detect the pose estimations of the person.
- We carry out this single pose estimation for each cropped bounding box in each frame, and finally annotate the video with the estimated pose landmarks.

In [4]:
def play_video(video_path: str):
    frac = 0.65 # scaling factor for display 
    display(
        Video(data=video_path, embed=True, height=int(720*frac), width=int(1280*frac))
    )

In [5]:
def detect_player_bbox(video_path:str) -> str:
    """
    Annotates video with object detection bounding boxes using YOLOv8 model.
    """
    
    video_name = video_path.split('/')[-1]
    folder_name = 'predict'
    
    # YOLOv8 Object Detection
    # you can edit the model if you want to use a different yolo8 model
    os.system(f"yolo predict model=yolov8x.pt source={video_path} save_txt=True")
    
    # path to latest output 
    max = 0
    for dir_name in os.listdir('/kaggle/working/runs/detect'):
        if dir_name!='predict':
            if int(dir_name[-1])>max:
                max = int(dir_name[-1])
    if max>0:
        folder_name += str(max)
    output_path = f'/kaggle/working/runs/detect/{folder_name}/labels'
    
    return output_path

In [6]:
def bbox_labels_to_dataframe(bbox_labels_path: str) -> pd.DataFrame:

    bbox_labels = {
        'video_name':[],
        'frame':[],
        'class_id':[],
        'center_x':[],
        'center_y':[],
        'width':[],
        'height':[]
    }

    for filename in os.listdir(bbox_labels_path):
        video_name = "_".join(filename.split('_')[0:3]) + '.mp4'
        frame = filename.split('_')[-1]
        frame = int(frame.split('.')[0])

        with open(bbox_labels_path + '/' + filename, 'r') as f:
            for line in f:
                line = line.split(" ")
                class_id = int(line[0])
                center_x = float(line[1])
                center_y = float(line[2])
                width = float(line[3])
                height = float(line[4])

                if class_id == 0: # if person
                    # append to dict
                    bbox_labels['video_name'].append(video_name)
                    bbox_labels['frame'].append(frame)
                    bbox_labels['class_id'].append(class_id)
                    bbox_labels['center_x'].append(center_x)
                    bbox_labels['center_y'].append(center_y)
                    bbox_labels['width'].append(width)
                    bbox_labels['height'].append(height)
                    
    
    return pd.DataFrame(bbox_labels)

In [7]:
def multi_pose_estimation(video_path:str, bbox_labels: pd.DataFrame, verbose=True) -> str:
    """
    Performs multi-shot multi-pose estimation by obtianing person bbox from YOLOv8
    and performing single pose estimation on the bbox crop.
    """
    
    # intializing mediapipe utils
    mp_drawing = mp.solutions.drawing_utils
    mp_drawing_styles = mp.solutions.drawing_styles
    mp_pose = mp.solutions.pose
    
    # video name
    video_name = video_path.split('/')[-1]

    # VideoCapture Object
    cap = cv2.VideoCapture(video_path)
    
    # video variables
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = bbox_labels['frame'].max()
    
    # VideoWriter Object
    output_path  = "labeled_" + video_name
    tmp_output_path = 'tmp_' + output_path
    out = cv2.VideoWriter(tmp_output_path, cv2.VideoWriter_fourcc(*'MP4V'), fps, (width, height))

    # check if camera opened successfully
    if (cap.isOpened()==False):
        print('Error opening video file!')

    # multipose estimation
    with mp_pose.Pose(
        min_detection_confidence = 0.8,
        min_tracking_confidence = 0.8) as pose:
        frame = 1
        while (cap.isOpened()):
            success, image = cap.read()
            if success:
                # selecting the frame
                bbox_set = bbox_labels.query('frame==@frame')
                # iterating through bboxs in the frame
                for idx, annot in bbox_set.iterrows():
                    bbox_center_x = annot['center_x'] * width
                    bbox_center_y = annot['center_y'] * height
                    bbox_width = annot['width'] * width
                    bbox_height = annot['height'] * height

                    # finding top-left and bottom-right bbox cooridnates
                    bbox_top_left_x = int(bbox_center_x - (bbox_width/2))
                    bbox_top_left_y = int(bbox_center_y - (bbox_height/2))
                    bbox_bottom_right_x = int(bbox_center_x + (bbox_width/2))
                    bbox_bottom_right_y = int(bbox_center_y + (bbox_height/2))

                    # cropping image to bbox
                    image_crop = image[bbox_top_left_y:bbox_bottom_right_y, bbox_top_left_x:bbox_bottom_right_x]

                    # pose estimation
                    # set image as not writeable to improve perfromance
                    image_crop.flags.writeable = False
                    image_crop = cv2.cvtColor(image_crop, cv2.COLOR_BGR2RGB)
                    results = pose.process(image_crop)

                    # transposing results to be drawn on the original image
                    if results.pose_landmarks != None:
                        for landmark in results.pose_landmarks.landmark:
                            landmark.x = ((abs(bbox_bottom_right_x - bbox_top_left_x) / width) * landmark.x) + (bbox_top_left_x/width)
                            landmark.y = ((abs(bbox_bottom_right_y - bbox_top_left_y) / height) * landmark.y) + (bbox_top_left_y/height)

                        # draw the pose annotations on the image
                        # set image as writeable
                        image.flags.writeable = True
                        mp_drawing.draw_landmarks(
                            image,
                            results.pose_landmarks,
                            mp_pose.POSE_CONNECTIONS,
                            landmark_drawing_spec = mp_drawing_styles.get_default_pose_landmarks_style())



                # save video
                out.write(image)
                if verbose:
                    print(f'Frame: {frame}/{total_frames}')
                frame += 1
            else:
                break


        cap.release()
        out.release()
        

    # Not all browsers support the codec, we will re-load the file at tmp_output_path
    # and convert to a codec that is more broadly readable using ffmpeg
    if os.path.exists(output_path):
        os.remove(output_path)
    subprocess.run(
            [
                "ffmpeg",
                "-i",
                tmp_output_path,
                "-crf",
                "18",
                "-preset",
                "veryfast",
                "-hide_banner",
                "-loglevel",
                "error",
                "-vcodec",
                "libx264",
                output_path,
            ]
        )
    os.remove(tmp_output_path)
    
    return output_path

In [8]:
video_path = '/kaggle/input/nfl-player-contact-detection/train/58168_003392_Endzone.mp4'
bbox_labels_path = detect_player_bbox(video_path)
bbox_labels = bbox_labels_to_dataframe(bbox_labels_path)

100%|██████████| 131M/131M [00:00<00:00, 181MB/s]

Ultralytics YOLOv8.0.22 🚀 Python-3.7.12 torch-1.11.0 CUDA:0 (Tesla P100-PCIE-16GB, 16281MiB)
YOLOv8x summary (fused): 268 layers, 68200608 parameters, 0 gradients, 257.8 GFLOPs
video 1/1 (1/711) /kaggle/input/nfl-player-contact-detection/train/58168_003392_Endzone.mp4: 384x640 20 persons, 27.4ms
video 1/1 (2/711) /kaggle/input/nfl-player-contact-detection/train/58168_003392_Endzone.mp4: 384x640 20 persons, 27.2ms
video 1/1 (3/711) /kaggle/input/nfl-player-contact-detection/train/58168_003392_Endzone.mp4: 384x640 20 persons, 27.0ms
video 1/1 (4/711) /kaggle/input/nfl-player-contact-detection/train/58168_003392_Endzone.mp4: 384x640 20 persons, 1 baseball glove, 27.1ms
video 1/1 (5/711) /kaggle/input/nfl-player-contact-detection/train/58168_003392_Endzone.mp4: 384x640 20 persons, 1 baseball glove, 26.3ms
video 1/1 (6/711) /kaggle/input/nfl-player-contact-detection/train/58168_003392_Endzone.mp4: 384x640 20 persons, 25.2ms
video 1/1 (7/711

In [9]:
output_path = multi_pose_estimation(video_path, bbox_labels)

OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Frame: 1/711
Frame: 2/711
Frame: 3/711
Frame: 4/711
Frame: 5/711
Frame: 6/711
Frame: 7/711
Frame: 8/711
Frame: 9/711
Frame: 10/711
Frame: 11/711
Frame: 12/711
Frame: 13/711
Frame: 14/711
Frame: 15/711
Frame: 16/711
Frame: 17/711
Frame: 18/711
Frame: 19/711
Frame: 20/711
Frame: 21/711
Frame: 22/711
Frame: 23/711
Frame: 24/711
Frame: 25/711
Frame: 26/711
Frame: 27/711
Frame: 28/711
Frame: 29/711
Frame: 30/711
Frame: 31/711
Frame: 32/711
Frame: 33/711
Frame: 34/711
Frame: 35/711
Frame: 36/711
Frame: 37/711
Frame: 38/711
Frame: 39/711
Frame: 40/711
Frame: 41/711
Frame: 42/711
Frame: 43/711
Frame: 44/711
Frame: 45/711
Frame: 46/711
Frame: 47/711
Frame: 48/711
Frame: 49/711
Frame: 50/711
Frame: 51/711
Frame: 52/711
Frame: 53/711
Frame: 54/711
Frame: 55/711
Frame: 56/711
Frame: 57/711
Frame: 58/711
Frame: 59/711
Frame: 60/711
Frame: 61/711
Frame: 62/711
Frame: 63/711
Frame: 64/711
Frame: 65/711
Frame: 66/711
Frame: 67/711
Frame: 68/711
Frame: 69/711
Frame: 70/711
Frame: 71/711
Frame: 72/711
F

In [10]:
play_video(output_path)

# References
- https://www.kaggle.com/code/dariussingh/player-contact-detection-eda
- https://www.kaggle.com/code/dariussingh/nfl-yolov8-object-detection-and-segmentation
- https://github.com/ultralytics/ultralytics
- https://google.github.io/mediapipe/solutions/pose